# CNGC6 (*Solanum lycopersicum*): reproducibility of the calculations

This notebook reproduces the calculations of manuscript ART 982. It runs on Google Colab (or any Linux system with Python 3.10+).

**Order:** 1) environment and software · 2) repository data · 3) QMEAN and membrane · 4) PPM 2 · 5) comparison with cryo-EM (RMSD and TM-score) · 6) interfaces · 7) pore: HOLE and MOLEonline · 8) secondary structure (DSSP) · 9) data tables · 10) summary.

**No manual uploads are needed.** The notebook clones the repository https://github.com/camilotayac/Three-Dimensional-Structure-of-the-Tomato-Solanum-lycopersicum-L.-CNGC6-Channel (branch `master`). The cryo-EM structures are downloaded from RCSB. The QMEAN files are read from `qmean/<MODEL>/mem_info.json` and `qmean/<MODEL>/score_file.json` inside the repository; if a folder is missing, that model is skipped.


## 1. Environment and software

In [2]:
# System programs and Python libraries
!apt-get -qq update > /dev/null
!apt-get -qq install -y gfortran dssp > /dev/null
!pip -q install MDAnalysis mdahole2 freesasa tmtools biopython openpyxl scipy
import os, shutil; print('mkdssp:', shutil.which('mkdssp'), '| gfortran:', shutil.which('gfortran'))

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
mkdssp: /usr/bin/mkdssp | gfortran: /usr/bin/gfortran


In [3]:
# 1a) Full repository (tetramer, monomer, pore, conserved_residues, evaluation)
!git clone -q --depth 1 https://github.com/camilotayac/Three-Dimensional-Structure-of-the-Tomato-Solanum-lycopersicum-L.-CNGC6-Channel.git repo
REPO='/content/repo'
!ls {REPO}
!du -sh {REPO}

CNGC6_reproducibilidad.ipynb  monomer		       QMEAN
conserved_residues	      pore		       README.md
IMG			      PPM2_results_CNGC6.xlsx  tetramer
157M	/content/repo


In [4]:
# 1b) Experimental structures from the PDB (RCSB): cryo-EM of CNGC1, CNGC5 and MtCNGC15
import urllib.request
PDB_IDS=['9J34','9J35','9J36','9KCV']
for pid in PDB_IDS:
    url=f'https://files.rcsb.org/download/{pid}.pdb'
    urllib.request.urlretrieve(url, f'/content/{pid}.pdb')
    print(pid, os.path.getsize(f'/content/{pid}.pdb')//1024, 'KB')
# MODELLER templates (reference only; not used in the main calculations)
for pid in ['5U6O','7NP4','8BX7','8EVA']:
    try:
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pid}.pdb', f'/content/{pid}.pdb')
    except Exception as e: print('no descargado', pid, e)
print('PDB descargados')

9J34 1424 KB
9J35 1464 KB
9J36 1465 KB
9KCV 1473 KB
PDB descargados


In [5]:
# HOLE 2 (compiled from source; the executable is hole2/exe/hole)
!git clone -q --depth 1 https://github.com/osmart/hole2.git hole2
%cd /content/hole2/src
!bash -c "source ../source.apache && make" > /dev/null 2>&1
%cd /content
HOLE_EXE='/content/hole2/exe/hole'
import os; print('HOLE:', os.path.exists(HOLE_EXE))

/content/hole2/src
/content
HOLE: True


In [6]:
# PPM 2 (immers) desde ppm-python (GPL-3). Compilado local; no es PPM 3.0.
!git clone -q --depth 1 https://github.com/examol-corp/ppm-python.git ppmpy
%cd /content/ppmpy
!make > /dev/null 2>&1
%cd /content
!cp /content/ppmpy/tests/examples/testA/res.lib /content/ 2>/dev/null
PPM_EXE='/content/ppmpy/immers'
print('immers:', os.path.exists(PPM_EXE))

/content/ppmpy
/content
immers: True


## 2. Common utilities

In [7]:
import numpy as np, json, math, itertools, re
from Bio import Align

AA={'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H',
    'ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W',
    'TYR':'Y','VAL':'V'}

def load_ca(pdb):
    ch={}
    for l in open(pdb):
        if l.startswith('ATOM') and l[12:16].strip()=='CA':
            c=l[21] if l[21].strip() else 'A'
            ch.setdefault(c,[]).append((int(l[22:26]),AA.get(l[17:20],'X'),
                                       np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])])))
    return ch

def seqmap(model_chain, exp_chain):
    s1=''.join(r[1] for r in model_chain); s2=''.join(r[1] for r in exp_chain)
    al=Align.PairwiseAligner(); al.mode='global'
    al.substitution_matrix=Align.substitution_matrices.load('BLOSUM62')
    al.open_gap_score=-10; al.extend_gap_score=-0.5
    a=al.align(s1,s2)[0]; pairs=[]
    for (a0,a1),(b0,b1) in zip(*a.aligned):
        for k in range(a1-a0): pairs.append((a0+k,b0+k))
    ident=sum(1 for i,j in pairs if s1[i]==s2[j])/len(s1)
    return pairs, ident

def kabsch(P,Q):
    # Returns the RMSD and the rotation matrix R such that (P-Pc) @ R + Qc ≈ Q
    Pc=P.mean(0); Qc=Q.mean(0)
    u,s,vt=np.linalg.svd((P-Pc).T@(Q-Qc))
    d=np.sign(np.linalg.det(u@vt))
    R=u@np.diag([1,1,d])@vt
    rmsd=np.sqrt(np.mean(np.sum(((P-Pc)@R-(Q-Qc))**2,axis=1)))
    return rmsd, R, Pc, Qc
print('funciones listas')

funciones listas


## 3. Tetramer models (repository) and QMEANBrane membrane

In [8]:
TETRA=['RAS02','RAS01','RIH01','RIH02','RAH01','RAH02','RIS01','RAH03','RIS02','RIS03']
for m in TETRA:
    assert os.path.exists(f'{REPO}/tetramer/{m}.pdb'), m
print('10 tetramers found')

10 tetramers found


In [9]:
# QMEANBrane summary (membrane_energy, width, axis and residues in the membrane).
# Requires mem_info.json and score_file.json per model in the repository folder qmean/<MODEL>/ (downloaded from the QMEAN server).
# Repository layout: qmean/<MODEL>/mem_info.json and qmean/<MODEL>/score_file.json (e.g. qmean/RAS02/mem_info.json).
def qmean_summary(model):
    d=f'{REPO}/qmean/{model}'
    m=json.load(open(f'{d}/mem_info.json')); s=json.load(open(f'{d}/score_file.json'))
    ax=np.array(m['membrane_axis']); tilt=math.degrees(math.acos(abs(ax[2])/np.linalg.norm(ax)))
    out={'model':model,'membrane_energy':m['membrane_energy'],'width':m['membrane_width'],
         'pos':m['membrane_pos'],'tilt_vs_z':tilt}
    for ch in 'ABCD':
        v=m['in_membrane'][ch]
        out[f'n_in_{ch}']=sum(1 for x in v if x is True)
    out['qmean4_z']=s['global_scores']['qmean4_z_score']
    return out

import glob
qm_models=[os.path.basename(p) for p in glob.glob(f'{REPO}/qmean/*') if os.path.isdir(p)]
print('QMEAN models available:', qm_models)
for mm in qm_models: print(qmean_summary(mm))

QMEAN models available: []


## 4. PPM 2: position in an implicit bilayer (transfer energy)

In [10]:
# Each model is run with the input ' 0 in <file>'. immers needs res.lib in the same directory.
import subprocess
def run_ppm(model):
    src=f'{REPO}/tetramer/{model}.pdb'
    shutil.copy(src, f'/content/{model}.pdb')
    inp=f' 0 in  {model}.pdb\n'
    with open(f'/content/{model}.in.inp','w') as f: f.write(inp)
    r=subprocess.run(f'{PPM_EXE} < {model}.in.inp', shell=True, cwd='/content', capture_output=True, text=True, timeout=3600)
    txt=r.stdout+r.stderr
    m=re.findall(r'emin=\s*(-?[\d.]+)\s+thickn=\s*([\d.]+)\+-\s*([\d.]+)\s+tilt=\s*([\d.]+)', txt)
    return float(m[-1][0]) if m else None
# Runs the ten models (~8 min each on one core; on Colab it can take ~1 h)
PPM={}
for mdl in TETRA:
    PPM[mdl]=run_ppm(mdl)
    print(mdl, PPM[mdl], flush=True)

RAS02 -150.3


KeyboardInterrupt: 

In [ ]:
# Random orientations for RAS01 and RAS02 (seed 42), as in the manuscript
rng=np.random.default_rng(42)
def randrot():
    q=rng.normal(size=4); q/=np.linalg.norm(q); a,b,c,d=q
    return np.array([[a*a+b*b-c*c-d*d,2*(b*c-a*d),2*(b*d+a*c)],
                     [2*(b*c+a*d),a*a-b*b+c*c-d*d,2*(c*d-a*b)],
                     [2*(b*d-a*c),2*(c*d+a*b),a*a-b*b-c*c+d*d]])
def rot_pdb(src,dst,R,t):
    out=[]
    for l in open(src):
        if l.startswith(('ATOM','HETATM')):
            xyz=np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])])
            n=R@xyz+t
            l=l[:30]+f"{n[0]:8.3f}{n[1]:8.3f}{n[2]:8.3f}"+l[54:]
        out.append(l)
    open(dst,'w').writelines(out)
for base in ['RAS02','RAS01']:
    for tag in ['a','b']:
        rot_pdb(f'{REPO}/tetramer/{base}.pdb',f'/content/{base}{tag}.pdb',randrot(),rng.uniform(-15,15,3))
print('orientations generated (run PPM on RAS02a, RAS02b, RAS01a, RAS01b with run_ppm)')

## 5. Comparison with the cryo-EM structures (RMSD and TM-score)

In [11]:
from tmtools import tm_align
EXP=['9J34','9J35','9J36','9KCV']
for f in EXP:
    assert os.path.exists(f'/content/{f}.pdb'), f'Sube {f}.pdb a Colab'
E={f:load_ca(f'/content/{f}.pdb') for f in EXP}
RES=[]
for m in TETRA:
    M=load_ca(f'{REPO}/tetramer/{m}.pdb'); ids=sorted(M)
    for f in EXP:
        ex=E[f]; eids=sorted(ex)
        pairs,ident=seqmap(M['A'],ex['A'])
        sel=[(i,j) for i,j in pairs if 92<=M['A'][i][0]<=393]          # dominio TM (S1–S6)
        best=None
        for perm in itertools.permutations(range(4)):
            P=np.vstack([np.array([M[ids[k]][i][2] for i,j in sel]) for k in range(4)])
            Q=np.vstack([np.array([ex[eids[perm[k]]][j][2] for i,j in sel]) for k in range(4)])
            rmsd,R,Pc,Qc=kabsch(P,Q)
            if best is None or rmsd<best[0]: best=(rmsd,perm)
        rmsd,perm=best
        SP=''.join(M[ids[k]][i][1] for k in range(4) for i,j in sel)
        SQ=''.join(ex[eids[perm[k]]][j][1] for k in range(4) for i,j in sel)
        P=np.vstack([np.array([M[ids[k]][i][2] for i,j in sel]) for k in range(4)])
        Q=np.vstack([np.array([ex[eids[perm[k]]][j][2] for i,j in sel]) for k in range(4)])
        r=tm_align(P,Q,SP,SQ)
        RES.append((m,f,round(ident*100,1),round(rmsd,2),round(float(r.tm_norm_chain2),3)))
        print(RES[-1], flush=True)

('RAS02', '9J34', 61.4, np.float64(3.15), 0.934)
('RAS02', '9J35', 47.2, np.float64(3.4), 0.928)
('RAS02', '9J36', 47.2, np.float64(4.08), 0.899)
('RAS02', '9KCV', 44.4, np.float64(4.9), 0.883)
('RAS01', '9J34', 61.4, np.float64(3.03), 0.94)
('RAS01', '9J35', 47.2, np.float64(3.88), 0.909)
('RAS01', '9J36', 47.2, np.float64(3.57), 0.923)
('RAS01', '9KCV', 44.4, np.float64(4.67), 0.918)
('RIH01', '9J34', 60.7, np.float64(15.83), 0.461)
('RIH01', '9J35', 46.7, np.float64(17.24), 0.43)
('RIH01', '9J36', 46.7, np.float64(16.97), 0.433)
('RIH01', '9KCV', 43.9, np.float64(16.38), 0.446)
('RIH02', '9J34', 60.7, np.float64(14.46), 0.495)
('RIH02', '9J35', 46.7, np.float64(16.02), 0.455)
('RIH02', '9J36', 46.7, np.float64(15.7), 0.459)
('RIH02', '9KCV', 43.9, np.float64(15.24), 0.475)
('RAH01', '9J34', 61.4, np.float64(16.51), 0.432)
('RAH01', '9J35', 47.2, np.float64(18.28), 0.404)
('RAH01', '9J36', 47.2, np.float64(18.02), 0.412)
('RAH01', '9KCV', 44.4, np.float64(17.36), 0.423)
('RAH02', '9J

## 6. Chain interfaces (FreeSASA)

In [12]:
import freesasa
def read_atoms(pdb):
    ch={}
    for l in open(pdb):
        if l.startswith('ATOM'):
            c=l[21] if l[21].strip() else 'A'; ch.setdefault(c,[]).append(l)
    return ch
def sasa_of(lines):
    s=freesasa.Structure()
    for l in lines:
        s.addAtom(l[12:16],l[17:20],l[22:26].strip(),l[21] if l[21].strip() else 'A',
                  float(l[30:38]),float(l[38:46]),float(l[46:54]))
    return freesasa.calc(s).totalArea()
def interface(pdb):
    ch=read_atoms(pdb); ids=sorted(ch)
    alone={c:sasa_of(ch[c]) for c in ids}
    pair={}
    for a,b in itertools.combinations(ids,2):
        pair[(a,b)]=(alone[a]+alone[b]-sasa_of(ch[a]+ch[b]))/2
    adj=[('A','B'),('B','C'),('C','D'),('A','D')]
    return float(np.mean([pair[p] for p in adj]))
IF={m: interface(f'{REPO}/tetramer/{m}.pdb') for m in TETRA}
for m in TETRA: print(m, round(IF[m],1), 'Å² per adjacent interface')

RAS02 2342.5 Å² per adjacent interface
RAS01 2707.3 Å² per adjacent interface
RIH01 1436.3 Å² per adjacent interface
RIH02 1790.4 Å² per adjacent interface
RAH01 1279.8 Å² per adjacent interface
RAH02 1085.4 Å² per adjacent interface
RIS01 2064.8 Å² per adjacent interface
RAH03 662.6 Å² per adjacent interface
RIS02 1866.5 Å² per adjacent interface
RIS03 1953.3 Å² per adjacent interface


## 7. Pore

In [13]:
# 7a) HOLE 2 on the tetramer, with the axis estimated by HOLE (Cα centroid and z direction)
import MDAnalysis as mda, warnings; warnings.filterwarnings('ignore')
from mdahole2.analysis import HoleAnalysis
u=mda.Universe(f'{REPO}/pore/mole_channels_9cMMsQSSp0CuVLJJ2Xzxog_10/homotetramero2.pdb')
ha=HoleAnalysis(u,select='protein',executable=HOLE_EXE); ha.run()
prof=ha.results.profiles[0]; r=prof['radius']; z=prof['rxn_coord']
k=r>0.3
print('HOLE: minimum radius %.2f Å at z=%.1f' % (r[k].min(), z[k][np.argmin(r[k])]))

HOLE: minimum radius 1.11 Å at z=4.4


In [14]:
# 7b) MOLEonline: layers with charge (Asp/Glu −1; Arg/Lys/His +1) and position relative to the membrane.
# Membrane: QMEANBrane of RAS02 (requires qmean/RAS02/mem_info.json in the repository).
base=f'{REPO}/pore/mole_channels_9cMMsQSSp0CuVLJJ2Xzxog_10/'
d=json.load(open(base+'json/data.json')); path=d['Channels']['Paths'][0]
prof_nodes=path['Profile']; layers=path['Layers']['LayersInfo']
print('path nodes:',len(prof_nodes),'| layers:',len(layers))
print('configuration:', d['Config']['ProbeRadius'], d['Config']['InteriorThreshold'], 'channel (InMembrane=%s)' % d['Config']['InMembrane'])

path nodes: 1402 | layers: 201
configuration: 14.44 3.0 channel (InMembrane=False)


In [17]:
import glob
# Transformation of homotetramero2 (residues 31–708) onto RAS02 by Kabsch on Cα atoms.
# Correct application: X_RAS02 = (X - Pc) @ R + Qc
H=load_ca(base+'homotetramero2.pdb'); T=load_ca(f'{REPO}/tetramer/RAS02.pdb')
hk=sorted(H); tk=sorted(T); perm=(0,3,2,1)
P=[];Q=[]
for k in range(4):
    for rr in range(31,709):
        P.append(H[hk[k]][rr-1][2] if False else dict((x[0],x[2]) for x in H[hk[k]])[rr])
        Q.append(dict((x[0],x[2]) for x in T[tk[perm[k]]])[rr])
P=np.array(P); Q=np.array(Q)
rmsd,R,Pc,Qc=kabsch(P,Q)
print('Superposition RMSD (Å):', round(rmsd,3))
assert rmsd<0.05, 'superposition does not match'

# Buscar recursivamente el archivo mem_info.json correspondiente al modelo RAS02 dentro de QMEAN
mem_files = glob.glob(f'{REPO}/QMEAN/**/RAS02/mem_info.json', recursive=True)
if not mem_files:
    mem_files = glob.glob(f'{REPO}/QMEAN/**/mem_info.json', recursive=True)

assert len(mem_files) > 0, "No se encontró el archivo mem_info.json en el repositorio"

m=json.load(open(mem_files[0]))
ax=np.array(m['membrane_axis']); ax=ax/np.linalg.norm(ax); pos=float(m['membrane_pos']); w=float(m['membrane_width'])
cn=[dict((x[0],x[2]) for x in T['A'])[rr]@ax for rr in range(479,611) if rr in dict((x[0],x[2]) for x in T['A'])]
sgn=1.0 if np.mean(cn)>pos else -1.0
def toU(xyz): return sgn*(((xyz-Pc)@R+Qc)@ax-pos)
half=w/2
def region(u): return 'extracellular' if u<-half else ('intracellular' if u>half else 'membrane')
nodes=[]
for p in prof_nodes:
    xyz=np.array([p['X'],p['Y'],p['Z']]); u_=float(toU(xyz))
    nodes.append(dict(dist=p['Distance'],u=u_,region=region(u_),radius=p['Radius']))
regs={r:sum(1 for n in nodes if n['region']==r) for r in ['extracellular','membrane','intracellular']}
print('nodes per region:', regs)
assert regs['extracellular']>0 and regs['intracellular']>0, 'check the transformation'

Superposition RMSD (Å): 0.001
nodes per region: {'extracellular': 238, 'membrane': 364, 'intracellular': 800}


In [18]:
# Layer charge and residues lining the pore, per region
dn=[n['dist'] for n in nodes]; un=[n['u'] for n in nodes]
rows=[]
for L in layers:
    g=L['LayerGeometry']; mid=(g['StartDistance']+g['EndDistance'])/2
    u_=float(np.interp(mid,dn,un)); res=[s for s in L['Residues'] if 'Backbone' not in s]
    acid=sorted(set(' '.join(s.split()[:2]) for s in res if s.split()[0] in ('ASP','GLU')))
    basic=sorted(set(' '.join(s.split()[:2]) for s in res if s.split()[0] in ('LYS','ARG','HIS')))
    rows.append(dict(u=u_,region=region(u_),charge=L['Properties']['Charge'],minr=g['MinRadius'],acid=acid,basic=basic))
for rg in ['extracellular','membrane','intracellular']:
    sub=[r for r in rows if r['region']==rg]
    if not sub: continue
    print(rg, 'layers',len(sub),'net charge',sum(r['charge'] for r in sub),
          '| acidic',sorted(set(a for r in sub for a in r['acid'])),
          '| basic',sorted(set(b for r in sub for b in r['basic'])))

extracellular layers 38 net charge -48 | acidic ['GLU 318', 'GLU 367'] | basic []
membrane layers 51 net charge 0 | acidic [] | basic []
intracellular layers 112 net charge -50 | acidic ['ASP 586', 'ASP 587', 'ASP 702', 'ASP 707', 'GLU 409', 'GLU 706'] | basic ['ARG 496', 'ARG 596', 'ARG 597', 'HIS 560', 'LYS 413', 'LYS 589']


## 8. Secondary structure of the CNBD (DSSP)

In [19]:
# DSSP on chain A of RAS02 (a HEADER line is required by mkdssp)
lines=[l for l in open(f'{REPO}/tetramer/RAS02.pdb') if l.startswith(('ATOM','HETATM')) and (l[21] if l[21].strip() else 'A')=='A']
with open('/content/chainA.pdb','w') as f:
    f.write('HEADER    CNGC6 CHAIN A\n'); f.writelines(lines); f.write('END\n')
!mkdssp --output-format dssp /content/chainA.pdb /content/chainA.dssp
L=open('/content/chainA.dssp').read().splitlines()
i=[k for k,l in enumerate(L) if l.startswith('  #  RESIDUE')][0]
ss={}
for l in L[i+1:]:
    if len(l)<17 or l[13]=='!': continue
    ss[int(l[5:10])]=l[16] if l[16]!=' ' else '-'
def runs(codes):
    out=[];st=None;prev=None
    for n in sorted(ss):
        if ss[n] in codes:
            st=n if st is None else st; prev=n
        elif st is not None: out.append((st,prev)); st=None
    if st is not None: out.append((st,prev))
    return out
print('CNBD α-helices (≥4 res):',[h for h in runs('HGI') if h[1]-h[0]>=3 and h[0]>=470 and h[1]<=620])
print('CNBD β-strands (≥3 res):',[s for s in runs('EB') if s[1]-s[0]>=2 and s[0]>=470 and s[1]<=620])

CNBD α-helices (≥4 res): [(485, 494), (551, 556), (585, 598), (600, 610)]
CNBD β-strands (≥3 res): [(496, 500), (505, 507), (515, 521), (524, 528), (538, 542), (571, 574), (578, 584)]


## 9. Data tables from the repository

In [20]:
import pandas as pd
pd.set_option('display.max_columns',None); pd.set_option('display.width',200)
# 9a) MODELLER and SWISS-MODEL templates (monomer/modeling.xlsx)
mod=pd.read_excel(f'{REPO}/monomer/modeling.xlsx', sheet_name=None)
for name,df in mod.items():
    print('== hoja:', name, df.shape); display(df.head(20))

== hoja: Hoja 1 (20, 12)


,CNGC5,Unnamed: 1,Unnamed: 2,Unnamed: 3,CNGC6,Unnamed: 5,Unnamed: 6,Unnamed: 7,CNGC15,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,Homología,NaN,NaN,NaN,Homología,NaN,NaN,NaN,Homología,NaN,NaN,NaN
1,NaN,Modelo,Plantilla,Tipo,NaN,Modelo,Plantilla,Tipo,NaN,Modelo,Plantilla,Tipo
2,swiss-model,1,7xuf.1.D,Cryo-EM,swiss-model,1,8eva.1.B,Cryo-EM,swiss-model,1,6pby.1.A,cryo-EM
3,NaN,2,A0A6P5TGM7.1.A,AlphaFold DB,NaN,2,I1JYT9.1.A,AlphaFold DB,NaN,2,A0A067LBF3.1.A,AlphaFold DB
4,MODELLER HHPRED,1,5U6O_D,NaN,MODELLER HHPRED,1,5U6O_D,NaN,MODELLER HHPRED,1,5U6O_D,NaN
5,NaN,2,7NP4_B,NaN,NaN,2,7NP4_B,NaN,NaN,2,7NP4_B,NaN
6,NaN,3,8BX7_D,NaN,NaN,3,8BX7_D,NaN,NaN,3,8BX7_D,NaN
7,NaN,4,Conjunto 1-3,NaN,NaN,4,Conjunto 1-3,NaN,NaN,4,Conjunto 1-3,NaN
8,NaN,5,conjunto 1 y 2,NaN,NaN,5,conjunto 1 y 2,NaN,NaN,5,conjunto 1 y 2,NaN
9,NaN,6,conjunto 1 y 3,NaN,NaN,6,conjunto 1 y 3,NaN,NaN,6,conjunto 1 y 3,NaN


In [21]:
# 9b) Evaluation of the 16 monomers (monomer/evaluation.xlsx, sheet 'Hoja 3')
ev=pd.read_excel(f'{REPO}/monomer/evaluation.xlsx', sheet_name='Hoja 3')
ev=ev.dropna(subset=[ev.columns[0]])
display(ev)
# ProQ3D-lDDT correction for MODELLER 01–07 using the ProQ3D server values (see Table S1)
fix={'MODELLER HHPRED 01':0.304,'MODELLER HHPRED 02':0.353,'MODELLER HHPRED 03':0.280,
     'MODELLER HHPRED 04':0.348,'MODELLER HHPRED 05':0.340,'MODELLER HHPRED 06':0.332,'MODELLER HHPRED 07':0.330}
ev['ProQ3D_corr']=[fix.get(str(n).strip(), v) for n,v in zip(ev.iloc[:,0], ev.iloc[:,5])]
def mm(s): return (s-s.min())/(s.max()-s.min())
ev['composite']=(mm(ev.iloc[:,1])+mm(ev.iloc[:,2])+mm(ev.iloc[:,3])+mm(ev['ProQ3D_corr']))/4
display(ev.sort_values('composite',ascending=False)[[ev.columns[0],'composite']])

,Modelos,ModFOLD9\nGlobal Model\nQuality Score,ModFOLD9\nConfidence and P-value,QMEANDisCo,QMEAN4,ProQ3D-lDDT
0,AlphaFoldDB,0.000353,0.6141,0.54,-1.56,0.609
1,IntFOLD7 01,0.000316,0.6200,0.54,-0.77,0.599
2,IntFOLD7 02,0.000420,0.6048,0.54,-1.60,0.616
3,IntFOLD7 03,0.000483,0.5974,0.53,-1.07,0.594
4,ESMFold,0.000892,0.5648,0.52,-2.53,0.581
5,RoseTTAFold2,0.001296,0.5449,0.45,-4.73,0.577
6,OmegaFold,0.002204,0.5166,0.51,-3.56,0.560
7,SWISS -MODEL 01,0.334600,0.2354,0.36,-8.30,0.200
8,SWISS -MODEL 02,0.000554,0.5901,0.52,-1.72,0.629
9,MODELLER HHPRED 01,0.153100,0.2879,0.40,-5.60,-5.600


,Modelos,composite
0,AlphaFoldDB,0.734538
2,IntFOLD7 02,0.732622
1,IntFOLD7 01,0.732517
8,SWISS -MODEL 02,0.702965
3,IntFOLD7 03,0.701149
4,ESMFold,0.658799
6,OmegaFold,0.602323
5,RoseTTAFold2,0.546613
10,MODELLER HHPRED 02,0.411912
15,MODELLER HHPRED 07,0.384015


In [22]:
# 9c) PPM 2 and QMEAN results from the analysis workbook (PPM2_results_CNGC6.xlsx, if present in the repository)
import glob
xls=glob.glob(f'{REPO}/**/PPM2*.xlsx', recursive=True)+glob.glob('/content/PPM2*.xlsx')
for f in xls:
    print('file:', f)
    for name,df in pd.read_excel(f, sheet_name=None).items():
        print('  hoja:', name); display(df.head(15))

file: /content/repo/PPM2_results_CNGC6.xlsx
  hoja: PPM2_results


,Model,Input orientation,Transfer energy ΔG (kcal/mol),Hydrophobic thickness (Å),Tilt (°),Reading
0,RAS02,Original,-150.3,20.8,2.0,NaN
1,RAS02a,Random rotation a,-138.6,20.8,3.0,NaN
2,RAS02b,Random rotation b,-152.7,21.2,1.0,NaN
3,RAS01,Original,-131.5,20.8,0.0,NaN
4,RAS01a,Random rotation a,-112.2,21.2,4.0,NaN
5,RAS01b,Random rotation b,-109.7,22.4,3.0,NaN
6,RAH01,Original,-1.8,0.4,76.0,NaN
7,RAH02,Original,-2.4,0.4,85.0,NaN
8,RAH03,Original,0.0,20.8,60.0,NaN
9,RIS01,Original,-1.6,0.4,73.0,NaN


  hoja: QMEAN_membrane


,Model,membrane_energy (QMEANBrane),Rank by energy (1 = most negative),Energy / energy of RAS02,Membrane width (Å),Membrane axis vs C4 axis (°),"Residues in membrane, chain A",Chain B,Chain C,Chain D,QMEAN4 (Z),Mean local score,ΔG PPM 2 (kcal/mol)
0,RAS02,-224777.67,NaN,NaN,28.0,0.5,175.0,172.0,170.0,175.0,-1.42,0.783,NaN
1,RAS01,-196026.78,NaN,NaN,27.0,0.5,168.0,168.0,166.0,167.0,-1.40,0.778,NaN
2,RIH01,-78428.62,NaN,NaN,25.0,0.3,184.0,184.0,184.0,186.0,0.47,0.795,NaN
3,RIH02,-73514.87,NaN,NaN,21.9,0.2,152.0,150.0,153.0,152.0,0.45,0.794,NaN
4,RAH01,-66178.09,NaN,NaN,27.0,0.4,196.0,198.0,197.0,198.0,-1.07,0.782,NaN
5,RAH02,-56163.45,NaN,NaN,22.0,1.4,163.0,172.0,165.0,164.0,-1.31,0.777,NaN
6,RIS01,-45594.10,NaN,NaN,10.0,1.9,77.0,84.0,84.0,81.0,0.65,0.797,NaN
7,RAH03,-36544.86,NaN,NaN,10.0,14.7,99.0,83.0,72.0,102.0,-1.04,0.775,NaN
8,RIS02,-25430.29,NaN,NaN,10.0,55.0,125.0,67.0,0.0,0.0,0.48,0.779,NaN
9,RIS03,-20885.27,NaN,NaN,10.0,11.2,98.0,77.0,75.0,112.0,0.47,0.791,NaN


  hoja: Interface_TMscore


,Model,Monomer,Docking program,"TM-score vs RAS02 (tetramer, TM-align)",Adjacent interface area (Å²),Diagonal interface area (Å²),Interface residues (per interface),Hydrogen bonds (per interface),Salt bridges (per interface),Steric clashes (4 interfaces),Total buried area (Å²),Total buried area (% of the sum of chains),Clashes per Å² of interface (×1000)
0,RAS02,AlphaFoldDB,SymmDock,1.0000,2342.5,6.8,100.0,50.0,17.0,384.0,18735.0,10.52,NaN
1,RAS01,AlphaFoldDB,SymmDock,0.9701,2707.3,78.2,119.0,41.0,14.0,328.0,21564.5,12.11,NaN
2,RIH01,IntFOLD7,HSYMDOCK,0.2788,1436.3,0.0,69.0,26.0,4.0,152.0,11490.1,6.87,NaN
3,RIH02,IntFOLD7,HSYMDOCK,0.2805,1790.4,1.7,84.0,29.0,6.0,284.0,14322.6,8.56,NaN
4,RAH01,AlphaFoldDB,HSYMDOCK,0.3539,1279.8,0.0,61.0,21.0,0.0,116.0,10238.3,5.75,NaN
5,RAH02,AlphaFoldDB,HSYMDOCK,0.3440,1085.4,0.0,38.0,23.0,4.0,152.0,8683.3,4.88,NaN
6,RIS01,IntFOLD7,SymmDock,0.2800,2064.8,161.9,81.0,21.0,4.0,180.0,16384.3,9.80,NaN
7,RAH03,AlphaFoldDB,HSYMDOCK,0.3576,662.6,0.0,31.0,7.0,0.0,104.0,5300.8,2.98,NaN
8,RIS02,IntFOLD7,SymmDock,0.2551,1866.5,231.1,88.0,43.0,11.0,336.0,14715.9,8.80,NaN
9,RIS03,IntFOLD7,SymmDock,0.2650,1953.3,168.2,83.0,31.0,6.0,252.0,15382.9,9.20,NaN


... (8 outputs left out)

  hoja: TM_segments


,Model,Chain,Number of TM residues,TM residues assigned by PPM
0,RAS02,A,32.0,"32-43,52-69,89-105,131-146,177,181-194,218-235..."
1,RAS02,B,32.0,"32-44,46,51-68,89,91-107,130-146,181-195,218-2..."
2,RAS02,C,30.0,"32-44,46,51-68,89,91-105,107,130-146,177,181-1..."
3,RAS02,D,29.0,"31-43,52-69,72,89-105,132-146,149,177-178,180-..."
4,RAS01,A,29.0,"32-43,46,51-69,89,92-107,130-146,181-195,218-2..."
5,RAS01,B,29.0,"32-43,46,51-69,89,92-107,130-146,181-195,218-2..."
6,RAS01,C,29.0,"32-43,46,51-69,89,92-107,130-146,181-195,218-2..."
7,RAS01,D,29.0,"32-43,46,51-69,89,92-107,130-146,181-195,218-2..."
8,NaN,NaN,NaN,NaN
9,Segments assigned by PPM 2 to the bilayer regi...,NaN,NaN,NaN


  hoja: Notes


,Analysis,Independent verification of membrane compatibility (response to R1-1 / R2-M3d)
0,Program,"PPM 2 (immers, Lomize et al.), built locally (..."
1,Reference,"Lomize MA, Pogozheva ID, Joo H, Mosberg HI, Lo..."
2,Input,The 10 tetramers in the tetramer folder of the...
3,Date,31 October 2026
4,Caveat 1,PPM 2 depends on the initial orientation: the ...
5,Caveat 2,The QMEANBrane values in the manuscript are a ...
6,QMEAN,Repeated with QMEAN 4.6.0 on 31-Oct-2026 (firs...


In [23]:
# 9d) Table of the ten tetramers: PPM 2, interface and TM-score relative to RAS02
tab=pd.DataFrame({'model':TETRA})
tab['interfaz_A2']=[round(IF[m],1) for m in TETRA]
tab['PPM2_dG']=[PPM.get(m) for m in TETRA] if 'PPM' in globals() else None
display(tab)

,model,interfaz_A2,PPM2_dG
0,RAS02,2342.5,-150.3
1,RAS01,2707.3,NaN
2,RIH01,1436.3,NaN
3,RIH02,1790.4,NaN
4,RAH01,1279.8,NaN
5,RAH02,1085.4,NaN
6,RIS01,2064.8,NaN
7,RAH03,662.6,NaN
8,RIS02,1866.5,NaN
9,RIS03,1953.3,NaN


In [24]:
# 9e) Pore residues per layer (MOLEonline; repository pore/.../csv/path_1_physchem.csv)
phys=pd.read_csv(base+'csv/path_1_physchem.csv')
print('layers in path_1_physchem.csv:', len(phys))
display(phys[['LayerId','Radius','FreeRadius','Charge','Hydropathy','Polarity']].head(25))

layers in path_1_physchem.csv: 201


,LayerId,Radius,FreeRadius,Charge,Hydropathy,Polarity
0,1,9.31,9.62,-2,-1.82,21.26
1,2,7.38,7.68,-2,-1.40,25.70
2,3,6.90,7.19,-2,-1.73,33.13
3,4,5.21,5.45,-2,-3.50,49.70
4,5,4.43,4.77,-2,-3.40,50.33
5,6,2.63,2.62,-1,-3.35,50.65
6,7,2.46,2.47,0,-1.80,27.49
7,8,2.35,2.46,0,-1.33,19.45
8,9,1.86,2.02,0,-1.80,27.49
9,10,1.86,2.00,0,-1.33,19.45


## 11. Article tables, generated in the same format

In [25]:
# Values published in the manuscript (for comparison). Columns follow the article order.
ARTICLE={
 'T2':{ # Table 2: ten tetramers
   'RAS02':dict(tm=1.0000,E=-224778,w=28.0,ax=0.5,ppm='−139 to −153'),
   'RAS01':dict(tm=0.9701,E=-196027,w=27.0,ax=0.5,ppm='−110 to −132'),
   'RIH01':dict(tm=0.2788,E=-78429,w=25.0,ax=0.3,ppm='−3.6'),
   'RIH02':dict(tm=0.2805,E=-73515,w=21.9,ax=0.2,ppm='−3.6'),
   'RAH01':dict(tm=0.3539,E=-66178,w=27.0,ax=0.4,ppm='−1.8'),
   'RAH02':dict(tm=0.3440,E=-56163,w=22.0,ax=1.4,ppm='−2.4'),
   'RIS01':dict(tm=0.2800,E=-45594,w=10.0,ax=1.9,ppm='−1.6'),
   'RAH03':dict(tm=0.3576,E=-36545,w=10.0,ax=14.7,ppm='0.0'),
   'RIS02':dict(tm=0.2551,E=-25430,w=10.0,ax=55.0,ppm='−1.0'),
   'RIS03':dict(tm=0.2650,E=-20885,w=10.0,ax=11.2,ppm='−2.5')},
 'T3':{ # Table 3: interfaces (adjacent area Å², residues, H-bonds, salt bridges, clashes, buried area)
   'RAS02':(2343,100,50,17,384,18735),'RAS01':(2707,119,41,14,328,21564),
   'RIH01':(1436,69,26,4,152,11490),'RIH02':(1790,84,29,6,284,14323),
   'RAH01':(1280,61,21,0,116,10238),'RAH02':(1085,38,23,4,152,8683),
   'RIS01':(2065,81,21,4,180,16384),'RAH03':(663,31,7,0,104,5301),
   'RIS02':(1867,88,43,11,336,14716),'RIS03':(1953,83,31,6,252,15383)},
}
print('article values loaded')

article values loaded


In [26]:
# Table functions: compute the same quantities as the manuscript and compare them with ARTICLE
def interface_full(pdb):
    ch=read_atoms(pdb); ids=sorted(ch)
    alone={c:sasa_of(ch[c]) for c in ids}
    pair={}; tot=sasa_of(sum((ch[c] for c in ids),[]))
    for a,b in itertools.combinations(ids,2):
        pair[(a,b)]=(alone[a]+alone[b]-sasa_of(ch[a]+ch[b]))/2
    adj=[('A','B'),('B','C'),('C','D'),('A','D')]
    res=[];hb=[];sb=[];clash=0
    from scipy.spatial import cKDTree
    xyz={c:np.array([[float(l[30:38]),float(l[38:46]),float(l[46:54])] for l in ch[c]]) for c in ids}
    for a,b in adj:
        ta=cKDTree(xyz[a]); tb=cKDTree(xyz[b])
        near=ta.query_ball_tree(tb,4.5)
        ra={int(ch[a][i][22:26]) for i,js in enumerate(near) if js}
        rb={int(ch[b][j][22:26]) for js in near for j in js}
        res.append(len(ra)+len(rb))
        d=ta.sparse_distance_matrix(tb,4.0,output_type='coo_matrix')
        for i,j,v in zip(d.row,d.col,d.data):
            A=ch[a][i]; B=ch[b][j]
            if v<2.2: clash+=1
            if A[76:78].strip() in ('N','O') and B[76:78].strip() in ('N','O') and 2.2<=v<=3.5: hb.append(1)
            acid=lambda L: L[17:20] in ('ASP','GLU') and L[12:16].strip() in ('OD1','OD2','OE1','OE2')
            base=lambda L: (L[17:20]=='LYS' and L[12:16].strip()=='NZ') or (L[17:20]=='ARG' and L[12:16].strip() in ('NH1','NH2','NE'))
            if (acid(A) and base(B)) or (acid(B) and base(A)): sb.append(1)
    area=float(np.mean([pair[p] for p in adj]))
    return dict(area=round(area),res=round(np.mean(res)/2),hb=round(len(hb)/4),sb=round(len(sb)/4),clash=clash,buried=round(sum(alone.values())-tot))
rows=[]
for m in TETRA:
    f=interface_full(f'{REPO}/tetramer/{m}.pdb'); a=ARTICLE['T3'][m]
    rows.append([m,f['area'],a[0],f['res'],a[1],f['hb'],a[2],f['sb'],a[3],f['clash'],a[4],f['buried'],a[5]])
t3=pd.DataFrame(rows,columns=['model','area_calc','area_art','res_calc','res_art','HB_calc','HB_art','SB_calc','SB_art','clashes_calc','clashes_art','buried_calc','buried_art'])
print('Table 3 (interfaces): calculated vs article'); display(t3)

Table 3 (interfaces): calculated vs article


,model,area_calc,area_art,res_calc,res_art,HB_calc,HB_art,SB_calc,SB_art,clashes_calc,clashes_art,buried_calc,buried_art
0,RAS02,2343,2343,50,100,0,50,17,17,384,384,18735,18735
1,RAS01,2707,2707,60,119,0,41,14,14,328,328,21564,21564
2,RIH01,1436,1436,34,69,0,26,4,4,152,152,11490,11490
3,RIH02,1790,1790,42,84,0,29,6,6,284,284,14323,14323
4,RAH01,1280,1280,30,61,0,21,0,0,116,116,10238,10238
5,RAH02,1085,1085,19,38,0,23,4,4,152,152,8683,8683
6,RIS01,2065,2065,40,81,0,21,4,4,180,180,16384,16384
7,RAH03,663,663,16,31,0,7,0,0,104,104,5301,5301
8,RIS02,1867,1867,44,88,0,43,11,11,336,336,14716,14716
9,RIS03,1953,1953,42,83,0,31,6,6,252,252,15383,15383


In [27]:
# Table 2: TM-score vs RAS02, membrane energy, width, axis and PPM 2
from tmtools import tm_align
T_ref=load_ca(f'{REPO}/tetramer/RAS02.pdb'); ids_ref=sorted(T_ref)
def tm_vs_ras02(m):
    M=load_ca(f'{REPO}/tetramer/{m}.pdb'); ids=sorted(M)
    pairs,_=seqmap(T_ref['A'],M['A'])
    sel=[(i,j) for i,j in pairs if 92<=T_ref['A'][i][0]<=393]
    P=np.vstack([np.array([T_ref[ids_ref[k]][i][2] for i,j in sel]) for k in range(4)])
    Q=np.vstack([np.array([M[ids[k]][j][2] for i,j in sel]) for k in range(4)])
    SP=''.join(T_ref[ids_ref[k]][i][1] for k in range(4) for i,j in sel)
    SQ=''.join(M[ids[k]][j][1] for k in range(4) for i,j in sel)
    return round(float(tm_align(P,Q,SP,SQ).tm_norm_chain2),4)
qm={}
for mm_ in TETRA:
    d_=f'{REPO}/qmean/{mm_}'
    if os.path.exists(f'{d_}/mem_info.json'): qm[mm_]=qmean_summary(mm_)
rows=[]
for m in TETRA:
    a=ARTICLE['T2'][m]
    q=qm.get(m,{})
    rows.append([m,round(tm_vs_ras02(m),4),a['tm'],q.get('membrane_energy'),a['E'],q.get('width'),a['w'],
                 round(q['tilt_vs_z'],1) if q else None,a['ax'],PPM.get(m) if 'PPM' in globals() else None,a['ppm']])
t2=pd.DataFrame(rows,columns=['model','TM_calc','TM_art','E_calc','E_art','width_calc','width_art','axis_calc','axis_art','PPM_calc','PPM_art'])
print('Table 2 (ten tetramers): calculated vs article'); display(t2)

Table 2 (ten tetramers): calculated vs article


,model,TM_calc,TM_art,E_calc,E_art,width_calc,width_art,axis_calc,axis_art,PPM_calc,PPM_art
0,RAS02,1.0000,1.0000,None,-224778,None,28.0,None,0.5,-150.3,−139 to −153
1,RAS01,0.9381,0.9701,None,-196027,None,27.0,None,0.5,NaN,−110 to −132
2,RIH01,0.4264,0.2788,None,-78429,None,25.0,None,0.3,NaN,−3.6
3,RIH02,0.4410,0.2805,None,-73515,None,21.9,None,0.2,NaN,−3.6
4,RAH01,0.4146,0.3539,None,-66178,None,27.0,None,0.4,NaN,−1.8
5,RAH02,0.3912,0.3440,None,-56163,None,22.0,None,1.4,NaN,−2.4
6,RIS01,0.3134,0.2800,None,-45594,None,10.0,None,1.9,NaN,−1.6
7,RAH03,0.3956,0.3576,None,-36545,None,10.0,None,14.7,NaN,0.0
8,RIS02,0.3034,0.2551,None,-25430,None,10.0,None,55.0,NaN,−1.0
9,RIS03,0.3135,0.2650,None,-20885,None,10.0,None,11.2,NaN,−2.5


In [28]:
# Table 4 and Table S1 (Table S1 values come from monomer/evaluation.xlsx, section 9b)
S4=pd.DataFrame({'residuo':['Lys213','Lys217','Arg227','Arg230'],
                 'conservation_%':['63 (K)','22 K / 19 R','99 (R)','56 (R)'],
                 'membrane_position_Å':[-18,-12,3,8],
                 'acidic_≤5Å':['none','Glu277','Asp138, Asp104','Asp185']})
print('Table 4 (basic residues of S4)'); display(S4)
print('Table S1: see cell 9b; the composite score should match the order in the article '
      '(IntFOLD7-01 0.983, AlphaFold DB 0.958, IntFOLD7-02 0.955).')

Table 4 (basic residues of S4)


,residuo,conservation_%,membrane_position_Å,acidic_≤5Å
0,Lys213,63 (K),-18,none
1,Lys217,22 K / 19 R,-12,Glu277
2,Arg227,99 (R),3,"Asp138, Asp104"
3,Arg230,56 (R),8,Asp185


Table S1: see cell 9b; the composite score should match the order in the article (IntFOLD7-01 0.983, AlphaFold DB 0.958, IntFOLD7-02 0.955).


## 12. Summary

In [29]:
import pandas as pd
print('Tetramers: PPM 2 energy (if run):', PPM)
print('Interface (Å² per interface):', {k:round(v,1) for k,v in IF.items()})
df=pd.DataFrame(RES, columns=['model','cryo-EM','identity_%','RMSD_TM_Å','TM_score'])
display(df)

Tetramers: PPM 2 energy (if run): {'RAS02': -150.3}
Interface (Å² per interface): {'RAS02': 2342.5, 'RAS01': 2707.3, 'RIH01': 1436.3, 'RIH02': 1790.4, 'RAH01': 1279.8, 'RAH02': 1085.4, 'RIS01': 2064.8, 'RAH03': 662.6, 'RIS02': 1866.5, 'RIS03': 1953.3}


,model,cryo-EM,identity_%,RMSD_TM_Å,TM_score
0,RAS02,9J34,61.4,3.15,0.934
1,RAS02,9J35,47.2,3.40,0.928
2,RAS02,9J36,47.2,4.08,0.899
3,RAS02,9KCV,44.4,4.90,0.883
4,RAS01,9J34,61.4,3.03,0.940
5,RAS01,9J35,47.2,3.88,0.909
6,RAS01,9J36,47.2,3.57,0.923
7,RAS01,9KCV,44.4,4.67,0.918
8,RIH01,9J34,60.7,15.83,0.461
9,RIH01,9J35,46.7,17.24,0.430
